<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
import numpy as np
import math
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

import warnings
warnings.filterwarnings("ignore")


In [3]:
"""Scratch dev/test module for VWAP Bounce core math. Logic gets ported into the real notebook
once verified here - not part of the final deliverable."""
from __future__ import annotations
import numpy as np
import pandas as pd

MAD_TO_SIGMA = 1.4826  # standard consistency constant for a normal distribution


def session_vwap(h, l, c, v):
    """VWAP(t) = cumsum((H+L+C)/3 * V) / cumsum(V), cumulative within the day.
    h/l/c/v: 1-D arrays, one row per minute, in time order for ONE ticker ONE day."""
    typical = (h + l + c) / 3.0
    cv = np.cumsum(typical * v)
    cvol = np.cumsum(v)
    with np.errstate(invalid="ignore", divide="ignore"):
        vwap = cv / cvol
    vwap[cvol <= 0] = np.nan
    return vwap


def expanding_vwap_sigma(c, vwap, min_bars=5):
    """Expanding (not rolling) std of (close - VWAP), NaN until min_bars bars have occurred.
    ddof=1, matching every other sigma in this codebase."""
    resid = c - vwap
    n = len(resid)
    out = np.full(n, np.nan)
    # expanding sum / sumsq for O(n) instead of O(n^2) recompute
    valid = np.isfinite(resid)
    x = np.where(valid, resid, 0.0)
    cnt = np.cumsum(valid.astype(np.int64))
    s1 = np.cumsum(x)
    s2 = np.cumsum(x * x)
    for t in range(n):
        k = cnt[t]
        if k >= min_bars:
            mean = s1[t] / k
            var = (s2[t] - k * mean * mean) / (k - 1)
            out[t] = np.sqrt(var) if var > 0 else np.nan
    return out


def tech_z(c, vwap, vwap_sigma, clip=8.0):
    with np.errstate(invalid="ignore", divide="ignore"):
        z = (c - vwap) / vwap_sigma
    return np.clip(z, -clip, clip)


def fit_beta_market_sector(stock_ret, market_ret, sector_ret):
    """OLS stock_ret ~ 1 + market_ret + sector_ret (WITH intercept, for an unbiased beta fit),
    but the intercept itself is NOT returned/used - `expected` downstream is purely
    beta_market*move + beta_sector*move, per the spec's own formula (no alpha term). Returns
    (beta_market, beta_sector, r2) or (nan,nan,nan) if too few points or singular design.
    If sector_ret is degenerate (e.g. identical to market_ret - ticker's own sector ETF
    resolved to the market ETF), sector term is dropped and beta_sector=0 is fit as a pure
    market regression instead, r2 from that reduced model."""
    y = np.asarray(stock_ret, float)
    xm = np.asarray(market_ret, float)
    xs = np.asarray(sector_ret, float)
    ok = np.isfinite(y) & np.isfinite(xm) & np.isfinite(xs)
    n = int(ok.sum())
    if n < 30:
        return np.nan, np.nan, np.nan
    y, xm, xs = y[ok], xm[ok], xs[ok]

    def _r2(y, yhat):
        ss_res = np.sum((y - yhat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # degeneracy check: sector regressor collinear with market (or with the constant)
    if np.std(xs) < 1e-12 or np.corrcoef(xm, xs)[0, 1] > 0.999:
        X = np.column_stack([np.ones(n), xm])
        coef, *_ = np.linalg.lstsq(X, y, rcond=None)
        bm = coef[1]
        yhat = X @ coef
        return float(bm), 0.0, float(_r2(y, yhat))

    X = np.column_stack([np.ones(n), xm, xs])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    yhat = X @ coef
    return float(coef[1]), float(coef[2]), float(_r2(y, yhat))


def mad_sigma(values):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < 5:
        return np.nan
    med = np.median(v)
    mad = np.median(np.abs(v - med))
    return float(mad * MAD_TO_SIGMA) if mad > 0 else np.nan


def pct_rank(values):
    """Percentile rank in [0,1], ties averaged - np.nan stays np.nan (not ranked)."""
    s = pd.Series(values, dtype=float)
    return s.rank(pct=True, na_option="keep").to_numpy()


def simulate_position(entry_idx, entry_px, fade_side, sigma_at_entry,
                       high, low, close, vwap, min_add_gap_bars, eod_idx,
                       add_mult=(1.0, 2.0), target_stop_mult=1.5):
    """One ticker's position lifecycle from entry_idx (the bar it fills AT, i.e. entry_px IS
    high[entry_idx..]/low/close[entry_idx]'s own open - caller passes entry_idx as the first bar
    INCLUDED in monitoring, entry_px as that bar's open) through target/stop/EOD, with up to two
    adverse add-ons. Bar-level first-touch using High/Low (not just Close) - "first-touch по 1 хв".

    Tie-break when a single bar's range contains BOTH the stop and the (that bar's own) VWAP
    target: STOP WINS (conservative backtest assumption - this is MY OWN choice, the spec does not
    resolve same-bar ambiguity; documented here and in the notebook's own comment).

    `sigma_at_entry` anchors BOTH the stop and the two add-on offsets for the position's entire
    life (frozen at signal time, not re-evaluated bar to bar) - target alone is dynamic (today's
    VWAP, which still moves after entry).

    min_add_gap_bars: minimum bars between fills (>=15 real minutes, caller converts to bar count -
    1 bar = 1 minute here, so this IS minutes).

    Returns a dict: exit_idx, exit_px, cls ("TARGET"/"STOP"/"EOD"), avg_entry_px, n_fills (1..3).
    """
    fills = [(entry_idx, entry_px)]
    avg_px = entry_px
    stop_px = entry_px - fade_side * target_stop_mult * sigma_at_entry
    add_done = 0
    last_fill_idx = entry_idx

    n = len(close)
    t = entry_idx
    while True:
        if t > eod_idx or t >= n:
            # ran off the end of the day's own data without a touch - close at the last bar seen
            last = min(t - 1, n - 1, eod_idx)
            return dict(exit_idx=last, exit_px=close[last], cls="EOD",
                        avg_entry_px=avg_px, n_fills=len(fills))

        hi, lo, vw = high[t], low[t], vwap[t]
        stop_touched = (hi >= stop_px) if fade_side < 0 else (lo <= stop_px)
        target_touched = (lo <= vw <= hi) if np.isfinite(vw) else False

        if stop_touched:
            return dict(exit_idx=t, exit_px=stop_px, cls="STOP", avg_entry_px=avg_px, n_fills=len(fills))
        if target_touched:
            return dict(exit_idx=t, exit_px=vw, cls="TARGET", avg_entry_px=avg_px, n_fills=len(fills))

        if t == eod_idx:
            return dict(exit_idx=t, exit_px=close[t], cls="EOD", avg_entry_px=avg_px, n_fills=len(fills))

        if add_done < len(add_mult) and (t - last_fill_idx) >= min_add_gap_bars:
            add_level = entry_px - fade_side * add_mult[add_done] * sigma_at_entry
            add_triggered = (hi >= add_level) if fade_side < 0 else (lo <= add_level)
            if add_triggered:
                fills.append((t, add_level))
                avg_px = float(np.mean([p for _, p in fills]))
                stop_px = avg_px - fade_side * target_stop_mult * sigma_at_entry
                add_done += 1
                last_fill_idx = t

        t += 1


In [4]:
"""Scratch dev/test for VWAP Bounce staging (daily aggregates + intraday regular-session minute
panel) directly off the real final.parquet. Ported into the real notebook once verified."""
from __future__ import annotations
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

SESSION_OPEN_MIN = 9 * 60 + 30   # 570, 09:30
SESSION_CLOSE_MIN = 16 * 60      # 960, 16:00 (exclusive - last bar is 15:59)


def stage_vwapbounce(input_path, daily_out_path, intraday_stage_dir, *, log_every_n_chunks=10):
    """One pass over final.parquet producing:
      - a single daily-aggregate parquet (ticker,date,o,h,l,c,v,prev_close,bench,sigma) - one
        row per (ticker,date), used for ADV20/ATR14/gap/beta-regression inputs.
      - one intraday parquet PER DATE under intraday_stage_dir (ticker,smin,o,h,l,c,v), REGULAR
        SESSION ONLY (09:30-16:00) - used for the online VWAP/tech_z/rel_z simulation. No
        midnight-crossing handling needed (unlike Reversal/Continuum) - VWAP Bounce never holds
        past the session's own close.
    """
    stage = Path(intraday_stage_dir)
    import shutil
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)

    daily_schema = pa.schema([
        ("ticker", pa.string()), ("date", pa.string()),
        ("o", pa.float32()), ("h", pa.float32()), ("l", pa.float32()), ("c", pa.float32()),
        ("v", pa.float64()), ("prev_close", pa.float64()), ("bench", pa.string()), ("sigma", pa.float64()),
    ])
    intraday_schema = pa.schema([
        ("ticker", pa.string()), ("smin", pa.int16()),
        ("o", pa.float32()), ("h", pa.float32()), ("l", pa.float32()), ("c", pa.float32()), ("v", pa.float64()),
    ])

    # In-memory accumulator for the daily aggregate, keyed (ticker,date) -> dict of running
    # o(first-by-smin)/h(max)/l(min)/c(last-by-smin)/v(sum)/prev_close/bench/sigma(last seen).
    # A (ticker,date) pair CAN legitimately straddle two row groups (CRACEN's own row groups are
    # NOT guaranteed to split exactly on a date boundary for every ticker) - writing one partial
    # row per row-group and re-combining them afterwards would get o/c wrong (need the true
    # first/last-by-TIME price, not a per-fragment "first row seen", and a plain re-groupby
    # can't recover that once smin is lost) - so this accumulates ACROSS the whole pass in memory
    # instead (~500k (ticker,date) pairs for this universe/window, comfortably small) and is
    # written once, fully correct, at the end.
    daily_acc: dict = {}
    intraday_writers = {}

    def _iw(date_str):
        if date_str not in intraday_writers:
            intraday_writers[date_str] = pq.ParquetWriter(str(stage / f"{date_str}.parquet"), intraday_schema, compression="zstd")
        return intraday_writers[date_str]

    pf = pq.ParquetFile(input_path)
    cols = ["ticker", "dt", "date", "o", "h", "l", "c", "v", "prev_close", "bench", "sigma"]
    cols = [c for c in cols if c in pf.schema.names]
    t0 = time.time()
    total_in = 0
    total_daily = 0
    total_intraday = 0
    print(f"START vwapbounce stage  file={input_path}")
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce")
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            smin = (dt.dt.hour.to_numpy("int32") * 60 + dt.dt.minute.to_numpy("int32")).astype("int32")

            # ---- regular-session intraday rows -> per-date file ----
            sess_mask = (smin >= SESSION_OPEN_MIN) & (smin < SESSION_CLOSE_MIN)
            if sess_mask.any():
                sdf = df.loc[sess_mask].copy()
                sdf["smin"] = smin[sess_mask].astype("int16")
                sdf = sdf.dropna(subset=["o", "h", "l", "c"])
                for d, part in sdf.groupby("date", sort=False):
                    out = part[["ticker", "smin", "o", "h", "l", "c", "v"]]
                    _iw(str(d)).write_table(pa.Table.from_pandas(out, schema=intraday_schema, preserve_index=False))
                    total_intraday += len(out)

            # ---- daily aggregate: REGULAR SESSION ONLY (same sess_mask as intraday) - "o" must be
            # open0 (the 09:30 print the spec itself defines), "c" the regular-session close that
            # pairs with "prev_close" (CRACEN's own fix rolls prev_close at 16:01, i.e. onto the
            # regular close) - pre/post-market prints would otherwise silently corrupt gap/ATR. ----
            dchunk = df.loc[sess_mask].assign(_smin=smin[sess_mask]).sort_values("_smin")
            if dchunk.empty:
                if (ci + 1) % log_every_n_chunks == 0:
                    print(f"[rg {ci+1:>4}/{pf.num_row_groups}] in={total_in:,} daily_pairs={len(daily_acc):,} "
                          f"intraday_rows={total_intraday:,} dates={len(intraday_writers)} elapsed={time.time()-t0:.1f}s")
                continue
            g = dchunk.groupby(["ticker", "date"], sort=False)
            o_first = g["o"].first()
            c_last = g["c"].last()
            h_max = g["h"].max()
            l_min = g["l"].min()
            v_sum = g["v"].sum()
            pc_last = g["prev_close"].last()
            bench_last = g["bench"].last()
            sigma_last = g["sigma"].last()
            for key in o_first.index:
                prev = daily_acc.get(key)
                o, c, h, l, v = o_first[key], c_last[key], h_max[key], l_min[key], v_sum[key]
                pc, b, sg = pc_last[key], bench_last[key], sigma_last[key]
                if prev is None:
                    daily_acc[key] = dict(o_first=o, c_last=c, h=h, l=l, v=v, prev_close=pc, bench=b, sigma=sg,
                                           _o_rg=ci, _c_rg=ci)
                else:
                    # o/c "ownership" tracked by which row-group was seen FIRST/LAST (row groups are
                    # processed in file order == time order), not by value - a later row group's
                    # chunk always wins "last" and a row group already holding the earliest "first"
                    # keeps it, regardless of price.
                    if ci < prev["_o_rg"]:
                        prev["o_first"] = o
                        prev["_o_rg"] = ci
                    if ci >= prev["_c_rg"]:
                        prev["c_last"] = c
                        prev["_c_rg"] = ci
                    prev["h"] = max(prev["h"], h) if pd.notna(prev["h"]) else h
                    prev["l"] = min(prev["l"], l) if pd.notna(prev["l"]) else l
                    prev["v"] = (prev["v"] or 0) + (v or 0)
                    if pd.notna(pc):
                        prev["prev_close"] = pc
                    if pd.notna(b):
                        prev["bench"] = b
                    if pd.notna(sg):
                        prev["sigma"] = sg
            total_daily = len(daily_acc)

            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci+1:>4}/{pf.num_row_groups}] in={total_in:,} daily_pairs={total_daily:,} "
                      f"intraday_rows={total_intraday:,} dates={len(intraday_writers)} elapsed={time.time()-t0:.1f}s")
    finally:
        for w in intraday_writers.values():
            w.close()

    rows = []
    for (tk, d), v in daily_acc.items():
        rows.append((tk, d, v["o_first"], v["h"], v["l"], v["c_last"], v["v"], v["prev_close"], v["bench"], v["sigma"]))
    daily_df = pd.DataFrame(rows, columns=["ticker", "date", "o", "h", "l", "c", "v", "prev_close", "bench", "sigma"])
    pq.write_table(pa.Table.from_pandas(daily_df, schema=daily_schema, preserve_index=False), str(daily_out_path))

    print(f"DONE vwapbounce stage in={total_in:,} daily_pairs={len(daily_df):,} intraday_rows={total_intraday:,} "
          f"dates={len(intraday_writers)} elapsed={time.time()-t0:.1f}s")



# -- earnings calendar (ported from CRACEN.ipynb/Reversal.ipynb's own /reports fetch - byte-for-byte
# the same retry wrapper and column-picking logic, just renamed) - "не торгувати в день реакції на
# earnings" needs a per-(ticker,date) boolean, built here as report_days_map: {ticker: {date_str,...}}.
# NOTE: this fetch needs the live DatumApi object (a notebook-global injected by the papermill run,
# exactly like CRACEN's own cells) - it is NOT reachable from a plain python shell, so this cell's own
# network path could not be exercised in isolated testing the way the rest of this notebook's math was;
# the retry/column-picking shape is copied from CRACEN.ipynb's own already-proven get_reports, not
# written fresh.
def _pick_first_existing(df, candidates):
    cols_l = {c.lower().strip(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_l:
            return cols_l[c.lower()]
    return None


def _datum_request_retry(endpoint, params, *, max_retries=6, backoff=0.5):
    import random, time
    for attempt in range(max_retries + 1):
        try:
            return DatumApi.data_request(endpoint, params)
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** attempt) * (0.9 + 0.2 * random.random()))


def fetch_report_days_map(tickers, start_date, end_date, max_workers=8, max_fail_ratio=0.15):
    """-> {ticker: set(report_date_str)} for every ticker in `tickers`. A ticker whose fetch fails
    simply gets no entry (its own days are never treated as earnings days) - same "fail open, not
    closed" choice Reversal.ipynb's own get_reports makes, since the strategy's job is to EXCLUDE
    earnings days, not require proof a day is report-free."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import pandas as pd

    fails = {"n": 0}

    def _one(t):
        try:
            params = {"ticker": str(t), "start_move_date": str(start_date)[:10], "end_move_date": str(end_date)[:10]}
            rep = _datum_request_retry("/reports", params)
            if rep is None or rep.empty:
                return t, set()
            dcol = _pick_first_existing(rep, ["report_date", "move_date", "date", "dt", "datetime",
                                               "published_at", "created_at"])
            if dcol is None:
                return t, set()
            dates = pd.to_datetime(rep[dcol], errors="coerce").dt.strftime("%Y-%m-%d").dropna()
            return t, set(dates.tolist())
        except Exception as ex:
            fails["n"] += 1
            print(f"WARN report fetch failed for {t}: {ex}")
            return t, set()

    out = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_one, t): t for t in tickers}
        for f in as_completed(futs):
            t, days = f.result()
            if days:
                out[t] = days

    if tickers and fails["n"] / len(tickers) > max_fail_ratio:
        print(f"WARN earnings calendar: {fails['n']}/{len(tickers)} tickers failed "
              f"({fails['n']/len(tickers)*100:.1f}% > {max_fail_ratio*100:.0f}%) - earnings exclusion is "
              f"running DEGRADED for this run (missing tickers simply never get excluded).")
    return out


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
"""Scratch dev/test: daily features (ADV20/ATR14/gap/beta_market/beta_sector/r2) from the staged
daily aggregate. Ported into the real notebook once verified."""
from __future__ import annotations
import numpy as np
import pandas as pd
from vwap_core import fit_beta_market_sector

BETA_LOOKBACK_DAYS = 60
BETA_MIN_DAYS = 30
MARKET_ETF = "SPY"


def build_daily_features(daily_df: pd.DataFrame) -> pd.DataFrame:
    """-> daily_df with added columns, ALL "as of yesterday" (shifted) except `gap`/`ret` which
    are the day's own: adv20_prev, atr14_prev, gap, ret, beta_market, beta_sector, r2.

    beta_market/beta_sector/r2 are fit on the trailing BETA_LOOKBACK_DAYS (60) of `ret` STRICTLY
    BEFORE the trading day they are attached to (min BETA_MIN_DAYS=30), so a ticker's "today" beta
    never sees today's own return - this is the walk-forward property the spec's own "рахуються
    на історії по ВЧОРАШНІЙ день включно (без зазирання в майбутнє)" requires.
    """
    d = daily_df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values(["ticker", "date"]).reset_index(drop=True)

    d["dollar_vol"] = d["c"] * d["v"]
    d["ret"] = d["c"] / d["prev_close"] - 1.0
    d["gap"] = d["o"] / d["prev_close"] - 1.0
    tr = np.maximum(d["h"] - d["l"], np.maximum((d["h"] - d["prev_close"]).abs(), (d["l"] - d["prev_close"]).abs()))
    d["_tr"] = tr

    g = d.groupby("ticker", sort=False)
    # shift(1) everywhere here: "as of yesterday's close" for use on TODAY's own row
    d["adv20_prev"] = g["dollar_vol"].transform(lambda s: s.shift(1).rolling(20, min_periods=20).mean())
    atr14 = g["_tr"].transform(lambda s: s.shift(1).rolling(14, min_periods=14).mean())
    d["atr14_prev_pct"] = atr14 / d["prev_close"] * 100.0

    # market (SPY) and each ticker's own sector-ETF (bench) daily return series, by date
    mkt_ret = d.loc[d["ticker"] == MARKET_ETF, ["date", "ret"]].set_index("date")["ret"]
    etf_ret = d[d["ticker"].isin(d["bench"].dropna().unique()) | (d["ticker"] == MARKET_ETF)]
    etf_ret_by_tk = {tk: g2.set_index("date")["ret"] for tk, g2 in etf_ret.groupby("ticker")}

    d["beta_market"] = np.nan
    d["beta_sector"] = np.nan
    d["r2"] = np.nan

    for tk, idx in g.groups.items():
        sub = d.loc[idx]
        bench_etf = sub["bench"].dropna()
        bench_etf = bench_etf.iloc[-1] if len(bench_etf) else None
        sec_ret = etf_ret_by_tk.get(bench_etf) if bench_etf else None
        if sec_ret is None:
            continue
        dates = sub["date"].to_numpy()
        rets = sub["ret"].to_numpy()
        bm = np.full(len(sub), np.nan)
        bs = np.full(len(sub), np.nan)
        r2 = np.full(len(sub), np.nan)
        for i, dt in enumerate(dates):
            lo = dt - pd.Timedelta(days=BETA_LOOKBACK_DAYS * 2)  # calendar-day slack for weekends/holidays
            window_dates = dates[(dates >= lo) & (dates < dt)]
            if len(window_dates) < BETA_MIN_DAYS:
                continue
            window_dates = window_dates[-BETA_LOOKBACK_DAYS:]
            stock_r = pd.Series(rets, index=dates).reindex(window_dates).to_numpy()
            mkt_r = mkt_ret.reindex(window_dates).to_numpy()
            sec_r = sec_ret.reindex(window_dates).to_numpy()
            bmi, bsi, r2i = fit_beta_market_sector(stock_r, mkt_r, sec_r)
            bm[i], bs[i], r2[i] = bmi, bsi, r2i
        d.loc[idx, "beta_market"] = bm
        d.loc[idx, "beta_sector"] = bs
        d.loc[idx, "r2"] = r2

    return d.drop(columns=["_tr"])


ModuleNotFoundError: No module named 'vwap_core'

In [ ]:
"""Scratch dev/test: the online cross-sectional simulation (div_sigma/volume-profile history,
candidate gating, wscore top-25%, position lifecycle) tying vwap_core/vwap_daily together.
Ported into the real notebook once verified."""
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path

from vwap_core import session_vwap, expanding_vwap_sigma, tech_z as tech_z_fn, mad_sigma, pct_rank, simulate_position

SESSION_OPEN_MIN = 570
SESSION_CLOSE_MIN = 960
BUCKET_MIN = 5
MIN_MINUTES_AFTER_OPEN = 30     # first decision point: 570+30=600 (10:00)
SIGMA_LOOKBACK_DAYS = 20
SIGMA_MIN_DAYS = 10             # pragmatic floor - spec says "~20", does not give a hard minimum
GATE_Z = 2.0
CLIP_Z = 8.0
TARGET_STOP_MULT = 1.5
ADD_MULTS = (1.0, 2.0)
MIN_ADD_GAP_MIN = 15
COST_RET = 0.0005               # 5 bps flat, per round trip
TOP_FRACTION = 0.25
WSCORE_W = dict(score=1.0, rvol=1.0, vwap_cross=-0.7, r2=-0.7, gap=0.5)

N_MIN = SESSION_CLOSE_MIN - SESSION_OPEN_MIN   # 390
N_BUCKETS = N_MIN // BUCKET_MIN                # 78
# RELATIVE indices into the per-ticker arrays (0 == 09:30, SESSION_OPEN_MIN already subtracted) -
# every array in this module (tz/rel_z/rvol/vwap_cross/o/h/l/c/vwap) is indexed this way; absolute
# minute-of-day is only reconstructed at the point a trade is RECORDED (+SESSION_OPEN_MIN).
DECISION_MINUTES = list(range(MIN_MINUTES_AFTER_OPEN, N_MIN, BUCKET_MIN))


def bucket_of(minute_of_day):
    return (minute_of_day - SESSION_OPEN_MIN) // BUCKET_MIN


def _load_day_panel(stage_dir, date_str, tickers_wanted):
    """-> dict ticker -> dict(o,h,l,c,v: float32[N_MIN] indexed by minute-of-day - SESSION_OPEN_MIN).
    Minutes with no print stay NaN (o/h/l/c) / 0 (v)."""
    f = Path(stage_dir) / f"{date_str}.parquet"
    if not f.exists():
        return {}
    df = pd.read_parquet(f)
    if tickers_wanted is not None:
        df = df[df["ticker"].isin(tickers_wanted)]
    if df.empty:
        return {}
    out = {}
    for tk, g in df.groupby("ticker", sort=False):
        o = np.full(N_MIN, np.nan, np.float32)
        h = np.full(N_MIN, np.nan, np.float32)
        l = np.full(N_MIN, np.nan, np.float32)
        c = np.full(N_MIN, np.nan, np.float32)
        v = np.zeros(N_MIN, np.float64)
        idx = (g["smin"].to_numpy() - SESSION_OPEN_MIN).astype(np.int64)
        keep = (idx >= 0) & (idx < N_MIN)
        idx = idx[keep]
        gk = g.loc[keep]
        o[idx] = gk["o"].to_numpy()
        h[idx] = gk["h"].to_numpy()
        l[idx] = gk["l"].to_numpy()
        c[idx] = gk["c"].to_numpy()
        v[idx] = gk["v"].to_numpy()
        out[tk] = dict(o=o, h=h, l=l, c=c, v=v)
    return out


def _ffill(a):
    """Forward-fill NaNs (a 1-min bar with no print carries the previous print's close forward -
    standard "last known price" convention, needed so VWAP/move series stay defined through a
    quiet minute instead of gapping to NaN)."""
    a = a.copy()
    mask = np.isnan(a)
    if mask.all():
        return a
    idx = np.where(~mask, np.arange(len(a)), 0)
    np.maximum.accumulate(idx, out=idx)
    return a[idx]


def _bucket_closes_and_vol(panel_row):
    """-> (bucket_close[N_BUCKETS], bucket_vol[N_BUCKETS]) - close = last valid close in the
    bucket (ffilled), vol = sum of 1-min volume in the bucket."""
    c = _ffill(panel_row["c"])
    v = panel_row["v"]
    bc = np.full(N_BUCKETS, np.nan)
    bv = np.zeros(N_BUCKETS)
    for b in range(N_BUCKETS):
        lo, hi = b * BUCKET_MIN, b * BUCKET_MIN + BUCKET_MIN
        seg_c = c[lo:hi]
        if np.isfinite(seg_c).any():
            bc[b] = seg_c[np.isfinite(seg_c)][-1]
        bv[b] = np.nansum(v[lo:hi])
    return bc, bv


class _History:
    """Rolling SIGMA_LOOKBACK_DAYS window of per-ticker bucket-close/bucket-volume arrays, one
    entry per trading day actually processed so far, evicted oldest-first. Built incrementally as
    the walk-forward loop advances - each day's own bucket arrays are computed ONCE (when that day
    is "today") and reused for every later day's lookback, never re-read from disk twice."""

    def __init__(self, maxlen):
        self.maxlen = maxlen
        self.dates = []
        self.bucket_close = {}   # date -> {ticker: array[N_BUCKETS]}
        self.bucket_vol = {}

    def push(self, date_str, bucket_close_by_tk, bucket_vol_by_tk):
        self.dates.append(date_str)
        self.bucket_close[date_str] = bucket_close_by_tk
        self.bucket_vol[date_str] = bucket_vol_by_tk
        if len(self.dates) > self.maxlen:
            old = self.dates.pop(0)
            del self.bucket_close[old]
            del self.bucket_vol[old]

    def div_sigma_and_volprofile(self, ticker, beta_market, beta_sector, mkt_bucket_close_by_date,
                                  sector_bucket_close_by_date):
        """-> (div_sigma[N_BUCKETS], vol_profile[N_BUCKETS]) from whatever history is buffered so
        far (NaN bucket where fewer than SIGMA_MIN_DAYS valid observations exist)."""
        div_obs = [[] for _ in range(N_BUCKETS)]
        vol_obs = [[] for _ in range(N_BUCKETS)]
        for d in self.dates:
            tk_c = self.bucket_close.get(d, {}).get(ticker)
            if tk_c is None:
                continue
            mkt_c = mkt_bucket_close_by_date.get(d)
            sec_c = sector_bucket_close_by_date.get(d)
            if mkt_c is None or sec_c is None:
                continue
            open0 = tk_c[0]
            mkt_open0 = mkt_c[0]
            sec_open0 = sec_c[0]
            if not (np.isfinite(open0) and np.isfinite(mkt_open0) and np.isfinite(sec_open0)):
                continue
            stock_move = tk_c / open0 - 1.0
            mkt_move = mkt_c / mkt_open0 - 1.0
            sec_move = sec_c / sec_open0 - 1.0
            expected = beta_market * mkt_move + beta_sector * sec_move
            div = stock_move - expected
            tk_v = self.bucket_vol.get(d, {}).get(ticker)
            for b in range(N_BUCKETS):
                if np.isfinite(div[b]):
                    div_obs[b].append(div[b])
                if tk_v is not None:
                    vol_obs[b].append(tk_v[b])
        div_sigma = np.full(N_BUCKETS, np.nan)
        vol_profile = np.full(N_BUCKETS, np.nan)
        for b in range(N_BUCKETS):
            if len(div_obs[b]) >= SIGMA_MIN_DAYS:
                div_sigma[b] = mad_sigma(div_obs[b])
            if len(vol_obs[b]) >= SIGMA_MIN_DAYS:
                vol_profile[b] = float(np.mean(vol_obs[b]))
        return div_sigma, vol_profile


def _ffill_ohlc(panel_row):
    c = _ffill(panel_row["c"])
    o = panel_row["o"].copy()
    h = panel_row["h"].copy()
    l = panel_row["l"].copy()
    bad = np.isnan(o)
    o[bad] = c[bad]
    bad = np.isnan(h)
    h[bad] = c[bad]
    bad = np.isnan(l)
    l[bad] = c[bad]
    return o, h, l, c


def _move_series(close_ffilled, open0):
    if not np.isfinite(open0) or open0 == 0:
        return np.full_like(close_ffilled, np.nan)
    return close_ffilled / open0 - 1.0


def simulate_day(date_str, panel, universe, ticker_meta, div_sigma_by_ticker, vol_profile_by_ticker,
                  earnings_today, mkt_ticker="SPY"):
    """One day's full cross-sectional VWAP-Bounce simulation.
    panel: {ticker: {o,h,l,c,v: float32[N_MIN]}} for every ticker needed (universe + SPY + every
    distinct bench ETF among `universe`), already built by `_load_day_panel`.
    ticker_meta: {ticker: dict(beta_market, beta_sector, r2, bench, gap, atr14_prev_pct, adv20_prev)}
    -> (trades: list[dict], candidates: list[dict])  "candidates" carries EVERY gate-passing
    reading (gate-free of the top-25% cut, like ALPHA's own universe elsewhere in this project) -
    used for this strategy's own alpha/lambda-equivalent stats.
    """
    if mkt_ticker not in panel:
        return [], [], []
    mkt_o, mkt_h, mkt_l, mkt_c = _ffill_ohlc(panel[mkt_ticker])
    mkt_move = _move_series(mkt_c, mkt_o[0] if np.isfinite(mkt_o[0]) else mkt_c[0])

    per_tk = {}
    for tk in universe:
        if tk not in panel or tk in earnings_today:
            continue
        meta = ticker_meta.get(tk)
        if meta is None or not np.isfinite(meta.get("beta_market", np.nan)):
            continue
        bench = meta.get("bench")
        if bench not in panel:
            continue
        adv20 = meta.get("adv20_prev", np.nan)
        if not np.isfinite(adv20) or adv20 < 25_000_000:
            continue

        o, h, l, c = _ffill_ohlc(panel[tk])
        open0 = o[0] if np.isfinite(o[0]) else c[0]
        if not np.isfinite(open0):
            continue
        vwap = session_vwap(h, l, c, panel[tk]["v"])
        vwap_sigma = expanding_vwap_sigma(c, vwap, min_bars=5)
        tz = tech_z_fn(c, vwap, vwap_sigma, clip=CLIP_Z)

        bo, bh, bl, bc = _ffill_ohlc(panel[bench])
        sector_move = _move_series(bc, bo[0] if np.isfinite(bo[0]) else bc[0])
        stock_move = _move_series(c, open0)
        expected = meta["beta_market"] * mkt_move + meta["beta_sector"] * sector_move
        div = stock_move - expected

        dsig = div_sigma_by_ticker.get(tk)
        rel_z = np.full(N_MIN, np.nan)
        if dsig is not None:
            bucket_idx = np.clip((np.arange(N_MIN)) // BUCKET_MIN, 0, N_BUCKETS - 1)
            dsig_per_min = dsig[bucket_idx]
            with np.errstate(invalid="ignore", divide="ignore"):
                rel_z = np.clip(div / dsig_per_min, -CLIP_Z, CLIP_Z)

        vprof = vol_profile_by_ticker.get(tk)
        rvol = np.full(N_MIN, np.nan)
        if vprof is not None:
            cum_v = np.cumsum(panel[tk]["v"])
            cum_profile_by_bucket = np.cumsum(np.where(np.isfinite(vprof), vprof, 0.0))
            bucket_idx = np.clip((np.arange(N_MIN)) // BUCKET_MIN, 0, N_BUCKETS - 1)
            denom = cum_profile_by_bucket[bucket_idx]
            with np.errstate(invalid="ignore", divide="ignore"):
                rvol = np.where(denom > 0, cum_v / denom, np.nan)

        sign_resid = (c - vwap) > 0
        cross = np.zeros(N_MIN, dtype=np.int32)
        flips = np.cumsum(np.diff(sign_resid.astype(np.int8), prepend=sign_resid[0]) != 0)
        cross[:] = flips

        per_tk[tk] = dict(o=o, h=h, l=l, c=c, vwap=vwap, vwap_sigma=vwap_sigma, tz=tz, rel_z=rel_z,
                           rvol=rvol, vwap_cross=cross, r2=meta.get("r2", np.nan), gap=meta.get("gap", np.nan))

    trades = []
    candidates = []
    signals = []   # gate-free: every (ticker, decision-minute) with both legs defined, regardless
                    # of threshold/sign-match/position-open status - this strategy's own ALPHA/
                    # LAMBDA source population, mirroring Reversal's gate-free SIGNAL table.
    open_positions = {}   # ticker -> resolution dict (once scheduled, never re-entered same day)

    for m in DECISION_MINUTES:
        round_candidates = []
        for tk, s in per_tk.items():
            tzm, rzm = s["tz"][m], s["rel_z"][m]
            if not (np.isfinite(tzm) and np.isfinite(rzm)):
                continue
            signals.append(dict(ticker=tk, date=date_str, m=m, score=min(abs(tzm), abs(rzm)),
                                 sign="pos" if tzm > 0 else "neg"))
            if tk in open_positions:
                continue
            if abs(tzm) < GATE_Z or abs(rzm) < GATE_Z:
                continue
            if np.sign(tzm) != np.sign(rzm):
                continue
            score = min(abs(tzm), abs(rzm))
            round_candidates.append(dict(ticker=tk, m=m, tz=tzm, rel_z=rzm, score=score,
                                          rvol=s["rvol"][m], vwap_cross=s["vwap_cross"][m],
                                          r2=s["r2"], gap=s["gap"]))
        if not round_candidates:
            continue

        cdf = pd.DataFrame(round_candidates)
        p_score = pct_rank(cdf["score"])
        p_rvol = pct_rank(cdf["rvol"])
        p_cross = pct_rank(cdf["vwap_cross"])
        p_r2 = pct_rank(cdf["r2"])
        p_gap = pct_rank(cdf["gap"].abs())
        wscore = (WSCORE_W["score"] * np.nan_to_num(p_score, nan=0.0)
                  + WSCORE_W["rvol"] * np.nan_to_num(p_rvol, nan=0.0)
                  + WSCORE_W["vwap_cross"] * np.nan_to_num(p_cross, nan=0.0)
                  + WSCORE_W["r2"] * np.nan_to_num(p_r2, nan=0.0)
                  + WSCORE_W["gap"] * np.nan_to_num(p_gap, nan=0.0))
        cdf["wscore"] = wscore
        wrank = pct_rank(cdf["wscore"])
        cdf["pass_top"] = wrank >= (1.0 - TOP_FRACTION)

        for row in cdf.itertuples():
            candidates.append(dict(ticker=row.ticker, date=date_str, m=row.m, score=row.score,
                                    sign="pos" if row.tz > 0 else "neg", wscore=row.wscore,
                                    passed_top=bool(row.pass_top)))
            if not row.pass_top:
                continue
            tk = row.ticker
            if tk in open_positions or (m + 1) >= N_MIN:
                continue
            s = per_tk[tk]
            entry_idx = m + 1
            entry_px = s["o"][entry_idx] if np.isfinite(s["o"][entry_idx]) else s["c"][entry_idx]
            sigma_at_entry = s["vwap_sigma"][m]
            if not (np.isfinite(entry_px) and np.isfinite(sigma_at_entry) and sigma_at_entry > 0):
                continue
            fade_side = -1.0 if row.tz > 0 else 1.0
            res = simulate_position(entry_idx, entry_px, fade_side, sigma_at_entry,
                                      s["h"], s["l"], s["c"], s["vwap"],
                                      min_add_gap_bars=MIN_ADD_GAP_MIN, eod_idx=N_MIN - 1,
                                      add_mult=ADD_MULTS, target_stop_mult=TARGET_STOP_MULT)
            gross = fade_side * (res["exit_px"] / res["avg_entry_px"] - 1.0)
            net = gross - COST_RET
            trades.append(dict(
                ticker=tk, date=date_str, sign="pos" if row.tz > 0 else "neg", cls=res["cls"],
                entry_dev=row.score, entry_minute=entry_idx + SESSION_OPEN_MIN,
                exit_minute=res["exit_idx"] + SESSION_OPEN_MIN, signal_minute=m + SESSION_OPEN_MIN,
                n_fills=res["n_fills"], pnl=gross * 100.0, net_pnl=net * 100.0,
            ))
            open_positions[tk] = res

    return trades, candidates, signals


def run_backtest(daily_feat_df, intraday_stage_dir, universe_tickers, min_adv20=25_000_000,
                  sigma_lookback_days=SIGMA_LOOKBACK_DAYS, report_days_map=None):
    """Walk-forward driver: advances date by date (ascending), maintaining the rolling _History
    buffer, calling simulate_day with yesterday's own frozen beta/adv20/atr14/gap (already shifted
    by build_daily_features). -> (trades_df, candidates_df)."""
    d = daily_feat_df[daily_feat_df["ticker"].isin(universe_tickers)].copy()
    all_dates = sorted(daily_feat_df["date"].dt.strftime("%Y-%m-%d").unique())
    hist = _History(maxlen=sigma_lookback_days)
    all_trades, all_candidates, all_signals = [], [], []

    # ticker -> bench ETF, resolved once per day from today's own daily_feat row (frozen choice is
    # fine - `bench` barely changes day to day in practice and the regression already uses the
    # SAME bench series it was fit against).
    by_date = {ds: g for ds, g in d.groupby(d["date"].dt.strftime("%Y-%m-%d"))}

    for date_str in all_dates:
        day_rows = by_date.get(date_str)
        etfs_needed = set()
        ticker_meta = {}
        if day_rows is not None:
            for r in day_rows.itertuples():
                if pd.isna(r.bench):
                    continue
                etfs_needed.add(r.bench)
                ticker_meta[r.ticker] = dict(beta_market=r.beta_market, beta_sector=r.beta_sector,
                                              r2=r.r2, bench=r.bench, gap=r.gap,
                                              atr14_prev_pct=r.atr14_prev_pct, adv20_prev=r.adv20_prev)
        today_universe = [tk for tk, m in ticker_meta.items() if np.isfinite(m["adv20_prev"]) and m["adv20_prev"] >= min_adv20]
        wanted = set(today_universe) | etfs_needed | {"SPY"}

        panel = _load_day_panel(intraday_stage_dir, date_str, wanted)

        # ---- 1) run TODAY's simulation using YESTERDAY's betas/div_sigma/vol_profile (both
        # already as-of-yesterday by construction: daily_feat_df's beta/adv/atr are pre-shifted,
        # and `hist` only ever contains days STRICTLY BEFORE today, pushed at the end of the
        # PREVIOUS iteration) ----
        earnings_today = {tk for tk in today_universe
                          if report_days_map and date_str in report_days_map.get(tk, ())} if report_days_map else set()
        if today_universe and panel:
            div_sigma_by_tk, vol_profile_by_tk = {}, {}
            for tk in today_universe:
                m = ticker_meta[tk]
                mkt_hist = {dt: hist.bucket_close.get(dt, {}).get("SPY") for dt in hist.dates}
                sec_hist = {dt: hist.bucket_close.get(dt, {}).get(m["bench"]) for dt in hist.dates}
                ds, vp = hist.div_sigma_and_volprofile(tk, m["beta_market"], m["beta_sector"], mkt_hist, sec_hist)
                div_sigma_by_tk[tk] = ds
                vol_profile_by_tk[tk] = vp
            trades, candidates, signals = simulate_day(date_str, panel, today_universe, ticker_meta,
                                               div_sigma_by_tk, vol_profile_by_tk, earnings_today=earnings_today)
            all_trades.extend(trades)
            all_candidates.extend(candidates)
            all_signals.extend(signals)

        # ---- 2) fold TODAY's own realized bars into the rolling history, for TOMORROW's lookback ----
        if panel:
            bc_today, bv_today = {}, {}
            for tk, row in panel.items():
                bc, bv = _bucket_closes_and_vol(row)
                bc_today[tk] = bc
                bv_today[tk] = bv
            hist.push(date_str, bc_today, bv_today)

    trades_df = pd.DataFrame(all_trades)
    candidates_df = pd.DataFrame(all_candidates)
    signals_df = pd.DataFrame(all_signals)
    return trades_df, candidates_df, signals_df


In [ ]:
from __future__ import annotations

# -- alpha, lambda, gamma, and the per-ticker rating from the trade/signal tables run_backtest produces ---
# VWAP Bounce has no exit-class dimension worth publishing (unlike Reversal/Continuum/OPG, which score
# per EXIT checkpoint): a position's `cls` (TARGET/STOP/EOD) is just how that one trade happened to end,
# not a separate situation a ticker could be independently good or bad at. So instead of Reversal's own
# per-(ticker,cls,sign) "ratings" array, this strategy publishes ONE rating per ticker (operator's own
# instruction, 2026-10: "потрібно зробити рейтинги для кожного тікера... показують співвідношення
# успішних ситуацій до усіх і те скільки ситуацій було" - one row per ticker, win_rate + total, fully
# pooled, no sign split either - confirmed explicitly over sign-split as the alternative). GAMMA is the
# same pooling applied to Reversal/CLO's own gamma algorithm (operator, same session: "перевір [CLO
# reversal] як тут реалізовані рейтинги і зроби так само" - gamma is the live-gate's own actual filter
# there, dropping it here would have been an unannounced simplification, not a requested one).
#
# compute_alpha/compute_lambda are adapted versions: VWAP Bounce's own gate-free reading is `score`
# (= min(|tech_z|,|rel_z|), ALWAYS non-negative - a magnitude, not a signed deviation like Reversal's
# Stack%), with direction carried in its own separate `sign` column instead of score's own sign - so
# ALPHA's split-by-sign step reads that column directly rather than inferring it from score>0/score<0.
import math

import numpy as np
import pandas as pd


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _modal_value(values, step=0.1, min_n=3):
    """The value a ticker gives MOST OFTEN, i.e. the mode of the reading, not its RMS - same
    definition every other strategy's own ALPHA uses. NaN below min_n readings."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    in_bin = np.abs(bins - best) < 1e-9
    return float(np.mean(v[in_bin]))


def compute_alpha(signals, step=0.1, min_n=3):
    """ALPHA, per ticker, split pos/neg: the modal `score` on each side of `sign`, over EVERY
    decision-minute reading the ticker produced - gate-free (not even the |z|>=2 two-leg gate),
    same universe principle as Reversal's own compute_alpha. -> DataFrame indexed by ticker,
    columns alpha_pos/alpha_neg."""
    if not len(signals):
        return pd.DataFrame(columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")
    out = {}
    for tk, g in signals.groupby("ticker"):
        out[tk] = (
            _modal_value(g.loc[g["sign"] == "pos", "score"], step, min_n),
            _modal_value(g.loc[g["sign"] == "neg", "score"], step, min_n),
        )
    return pd.DataFrame.from_dict(out, orient="index", columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")


def compute_lambda(signals, min_n=2):
    """LAMBDA (λ), per ticker: sample std (ddof=1) of `score` across EVERY decision-minute
    reading, gate-free, same population as ALPHA - this strategy's own adaptation of Reversal's
    "dispersion of the raw triggering metric" idea (VWAP Bounce has no single clean per-session
    "entry print" the way a Stack%-based strategy does, so `score`, the same magnitude ALPHA's
    own modal value is taken from, stands in for it here). NaN below min_n readings."""
    if not len(signals):
        return pd.Series(dtype=float, name="lambda").rename_axis("ticker")
    counts = signals.groupby("ticker")["score"].transform("count")
    out = signals[counts >= min_n].groupby("ticker")["score"].std(ddof=1)
    return out.rename("lambda").rename_axis("ticker")


def compute_ticker_rating(trades, min_n=1):
    """THE rating for this strategy: per ticker, POOLED across every cls and every sign - "scope
    успішних ситуацій до усіх і скільки їх було" is exactly total/wins/win_rate below; win_rate_lb/
    mean_pnl/pnl_lb/mean_net_pnl ride along as the same Wilson/t-test lower bounds every other
    strategy's own ratings already carry, not an extra ask.

    CAUTION (the operator was told this in-chat before it was built): ranking tickers BY this
    same-sample pnl_lb/win_rate and then judging them on THAT SAME sample is in-sample selection -
    a ticker with 5/5 wins at n=5 is a small-sample coin flip, not proven edge. This is only
    trustworthy computed WALK-FORWARD (fit on days up to T, judged on days after T) over a long
    history - exactly gamma's own frozen-snapshot discipline elsewhere in this project, just
    pooled across cls/sign instead of split by them. Until a long enough history exists, a
    high-ranked ticker here is a CANDIDATE to watch, not a validated live filter.
    """
    cols = ["ticker", "total", "wins", "win_rate", "win_rate_lb", "mean_pnl", "median_pnl", "pnl_lb"]
    if "net_pnl" in trades.columns:
        cols.append("mean_net_pnl")
    if not len(trades):
        return pd.DataFrame(columns=cols).set_index("ticker")
    g = trades.groupby("ticker")
    out = g["pnl"].agg(total="size", mean_pnl="mean", median_pnl="median", sd_pnl="std").reset_index()
    out["wins"] = g["pnl"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out = out[out["total"] >= min_n].copy()
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["total"])
        out["pnl_lb"] = np.where(out["total"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    if "net_pnl" in trades.columns:
        mean_net = trades.groupby("ticker")["net_pnl"].mean()
        out["mean_net_pnl"] = out["ticker"].map(mean_net)
    return out[cols].set_index("ticker")


def compute_gamma(trades, cfg):
    """GAMMA, per ticker - POOLED across every cls and sign, same reasoning as
    compute_ticker_rating: the LOWEST |score| level L >= gamma_floor at which trades reaching
    score >= L have a positive 95% lower bound on mean pnl, walking up from the floor in
    gamma_step increments, stopping at the first level that qualifies OR the first level with
    too little evidence (< gamma_min_n). Byte-for-byte Reversal/CLO's own compute_gamma algorithm
    (see that notebook's own version) - the ONLY change is the groupby key: ["ticker"] here
    instead of ["ticker","cls","sign"], because pooling is what makes gamma_min_n reachable at
    all in a short history (a ticker's OWN 8+ trades, not 8+ within one of its 6 tiny cells).
    None when nothing clears the bar for that ticker. -> DataFrame indexed by ticker."""
    cols = ["ticker", "gamma", "gamma_n", "gamma_mean", "gamma_rate"]
    if not len(trades):
        return pd.DataFrame(columns=cols).set_index("ticker")
    rows = []
    for tk, g in trades.groupby("ticker"):
        dev = np.abs(g["entry_dev"].to_numpy())
        pnl = g["pnl"].to_numpy()
        lvl = float(cfg["gamma_floor"])
        found = None
        while lvl <= cfg["gamma_max"] + 1e-9:
            m = dev >= lvl - 1e-9
            n = int(m.sum())
            if n < cfg["gamma_min_n"]:
                break
            mp = float(pnl[m].mean())
            se = float(pnl[m].std(ddof=1)) / math.sqrt(n) if n > 1 else math.inf
            lb = mp - cfg["gamma_z"] * se
            if lb > 0:
                found = (lvl, n, mp, float((pnl[m] > 0).mean()))
                break
            lvl = round(lvl + cfg["gamma_step"], 6)
        if found is not None:
            rows.append({"ticker": tk, "gamma": found[0], "gamma_n": found[1],
                        "gamma_mean": found[2], "gamma_rate": found[3]})
    return pd.DataFrame(rows, columns=cols).set_index("ticker")


In [ ]:
from pathlib import Path
import os, gzip, json, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# -- runner -----------------------------------------------------------------------------------------------
# Brand-new strategy, no existing backend to match byte-for-byte (unlike OPG-reversal, which already had
# a waiting C# stub) - "vwapbounce" follows the SAME no-separator-lowercase convention every other
# strategy's own folder name uses, so a future backend port has an obvious name to reuse.
FINAL_PATH, OUT_DIR = _resolve_orion_paths("vwapbounce")

STAGE_DIR = OUT_DIR.parent.parent / "_work" / "vwapbounce_stage"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
INTRADAY_STAGE_DIR = STAGE_DIR / "intraday"
DAILY_STAGE_PATH = STAGE_DIR / "daily.parquet"

_stage_ok = (DAILY_STAGE_PATH.exists() and INTRADAY_STAGE_DIR.exists()
             and any(INTRADAY_STAGE_DIR.glob("*.parquet"))
             and DAILY_STAGE_PATH.stat().st_mtime >= FINAL_PATH.stat().st_mtime)
if _stage_ok:
    print("stage is at least as new as final.parquet - reusing", STAGE_DIR)
else:
    STAGE_DIR.mkdir(parents=True, exist_ok=True)
    stage_vwapbounce(str(FINAL_PATH), str(DAILY_STAGE_PATH), str(INTRADAY_STAGE_DIR))

daily_raw = pd.read_parquet(DAILY_STAGE_PATH)
daily_feat = build_daily_features(daily_raw)

# sigma: READ from final.parquet's own column (via the daily stage), same "latest non-null wins"
# convention every other strategy's own stage uses - a ticker property, not computed here.
sigma_by_ticker = (
    daily_raw.dropna(subset=["sigma"])
    .sort_values(["ticker", "date"])
    .groupby("ticker")["sigma"].last()
    .to_dict()
)

# universe: every ticker that is NOT itself someone's benchmark ETF and not SPY - VWAP Bounce trades
# STOCKS against market+sector, trading an ETF against itself would be circular.
# DYNAMIC (whatever any ticker's own 'bench' resolved to - catches sector ETFs CRACEN's own
# correlation picker actually chose, e.g. ARKK) UNION a STATIC safety net of confirmed market/
# sector ETF tickers found in final.parquet by a full-file scan (2026-10-04) - needed because an
# ETF that LOSES every sector's own best-fit contest (verified real case: XLI, XLK - neither was
# ever anyone's 'bench' in this snapshot, yet both are plainly ETFs, not stocks) would otherwise
# slip into the tradeable universe undetected by the dynamic rule alone. Residual risk: an ETF
# outside BOTH this list and every ticker's own bench could still slip through - this is a known
# limitation, not a closed guarantee.
KNOWN_ETF_TICKERS = {
    "SPY", "QQQ", "IWM", "DIA", "XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE",
    "XLU", "XLV", "XLY", "XBI", "KRE", "XRT", "SOXX", "GDX", "KWEB", "COPX", "IGV", "IBIT", "NASA",
}
etf_set = set(daily_raw["bench"].dropna().unique()) | {"SPY"} | KNOWN_ETF_TICKERS
UNIVERSE = sorted(set(daily_raw["ticker"].unique()) - etf_set)
print(f"universe (ex-ETF): {len(UNIVERSE):,} tickers")

try:
    start_date = daily_raw["date"].min()
    end_date = daily_raw["date"].max()
    report_days_map = fetch_report_days_map(UNIVERSE, start_date, end_date)
    print(f"earnings calendar: {len(report_days_map):,} tickers with at least one report day")
except Exception as ex:
    print(f"WARN: earnings calendar fetch unavailable ({ex}) - running WITHOUT earnings exclusion "
          f"this run (every candidate day is treated as non-earnings).")
    report_days_map = None

trades, candidates, signals = run_backtest(
    daily_feat, str(INTRADAY_STAGE_DIR), UNIVERSE,
    min_adv20=25_000_000, sigma_lookback_days=20, report_days_map=report_days_map,
)
print(f"trades: {len(trades):,} | candidates(gate-passed): {len(candidates):,} | signals(gate-free): {len(signals):,} "
      f"| tickers with a trade: {trades['ticker'].nunique() if len(trades) else 0}")

CFG = dict(gamma_floor=2.0, gamma_step=0.25, gamma_max=8.0, gamma_min_n=8, gamma_z=1.96)

alpha = compute_alpha(signals)
lambda_by_ticker = compute_lambda(signals)
overall = compute_ticker_rating(trades)
gamma = compute_gamma(trades, CFG)
overall = overall.join(gamma, how="left")
overall.to_csv(OUT_DIR / "vwapbounce_ratings.csv")

trade_dates_sorted = sorted(trades["date"].unique().tolist()) if len(trades) else []
recent_dates = trade_dates_sorted[-255:]
date_index = {d: i for i, d in enumerate(recent_dates)}
WINDOWS_DAYS = [5, 20, 40, 65]


def _lambda_of(tk):
    return None if tk not in lambda_by_ticker.index else round(float(lambda_by_ticker[tk]), 4)


def _rating_of(tk, col):
    if tk not in overall.index:
        return None
    v = overall.loc[tk, col]
    return None if pd.isna(v) else round(float(v), 4)


def _ticker_rating_fields(tk):
    # No "cls"/"sign" split (see this notebook's own compute_ticker_rating doc comment for why) -
    # one flat set of fields per ticker, exactly "скільки ситуацій було" (total) and "співвідношення
    # успішних до усіх" (win_rate), plus the same Wilson/t-test lower bounds every other strategy's
    # own ratings already carry.
    return {
        "total": _rating_of(tk, "total"), "wins": None if tk not in overall.index else int(overall.loc[tk, "wins"]),
        "win_rate": _rating_of(tk, "win_rate"), "win_rate_lb": _rating_of(tk, "win_rate_lb"),
        "mean_pnl": _rating_of(tk, "mean_pnl"), "pnl_lb": _rating_of(tk, "pnl_lb"),
        "mean_net_pnl": _rating_of(tk, "mean_net_pnl"),
        "gamma": _rating_of(tk, "gamma"),
        "gamma_n": None if tk not in overall.index or pd.isna(overall.loc[tk, "gamma_n"]) else int(overall.loc[tk, "gamma_n"]),
        "gamma_mean": _rating_of(tk, "gamma_mean"), "gamma_rate": _rating_of(tk, "gamma_rate"),
    }


def _alpha_side(tk, col):
    if tk not in alpha.index:
        return None
    v = alpha.loc[tk, col]
    return None if pd.isna(v) else round(float(v), 4)


ROLLING_PATH = OUT_DIR / "vwapbounce_rolling_perf.json.gz"
rows = []
for tk, g in trades.groupby("ticker") if len(trades) else []:
    row = {
        "ticker": tk,
        "alpha_pos": _alpha_side(tk, "alpha_pos"), "alpha_neg": _alpha_side(tk, "alpha_neg"),
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk),
        "trades": [{
            # `cls` stays on each INDIVIDUAL trade (TARGET/STOP/EOD is just how that one trade
            # ended, a log fact) - only the AGGREGATE rating above drops cls/sign, per the
            # operator's own instruction.
            "cls": r["cls"], "sign": r["sign"], "date": date_index.get(r["date"]),
            "entry_dev": round(float(r["entry_dev"]), 4), "pnl_pct": round(float(r["pnl"]), 4),
            "net_pnl_pct": round(float(r["net_pnl"]), 4), "n_fills": int(r["n_fills"]),
            "status": "win" if r["pnl"] > 0 else "loss",
            "signal_minute_idx": int(r["signal_minute"]), "entry_minute_idx": int(r["entry_minute"]),
            "exit_minute_idx": int(r["exit_minute"]),
        } for _, r in g.iterrows() if r["date"] in date_index],
    }
    row.update(_ticker_rating_fields(tk))
    rows.append(row)
seen = {r["ticker"] for r in rows}
for tk in alpha.index:
    if tk in seen:
        continue
    ap, an = _alpha_side(tk, "alpha_pos"), _alpha_side(tk, "alpha_neg")
    if ap is None and an is None:
        continue
    row = {
        "ticker": tk, "alpha_pos": ap, "alpha_neg": an,
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk), "trades": [],
    }
    row.update(_ticker_rating_fields(tk))
    rows.append(row)

with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
    json.dump({
        "meta": {
            "version": "vwapbounce-rolling-perf-v1",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent_dates[-1] if recent_dates else None,
            "sessions_seen": len(recent_dates),
            "windows_days": WINDOWS_DAYS,
            "recent_dates": recent_dates,
            "position_usd": 1000,
            "gate_z": GATE_Z,
            "pnl_basis": "gross = fade_side*(exit/avg_entry_price - 1), fade_side=-sign(tech_z) (FADE: "
                        "short an extreme-high VWAP deviation, long an extreme-low one); entry = next "
                        "1-min bar's open after a cross-sectional top-25% wscore pass; exit = first-touch "
                        "VWAP target (cls=TARGET), 1.5x vwap_sigma stop (cls=STOP, frozen at signal time, "
                        "re-centered on the average fill after an add-on), or end-of-session close "
                        "(cls=EOD). Same-bar stop+target ties resolve to STOP (this notebook's own "
                        "conservative choice, not specified in the source spec). net_pnl_pct in each "
                        "trade already deducts a flat 5bps round-trip cost; the top-level pnl_pct/mean_pnl "
                        "figures are GROSS, matching every other strategy's own 'pnl' convention.",
            "rating_basis": "total/wins/win_rate(+Wilson LB)/mean_pnl(+95% LB)/mean_net_pnl/gamma(+gamma_n/"
                        "gamma_mean/gamma_rate), POOLED across every cls and sign for that ticker "
                        "(operator's own instruction, 2026-10: no per-class split here, one number per "
                        "ticker showing success-ratio + sample size) - gamma is the SAME lowest-qualifying-"
                        "score-threshold algorithm Reversal/CLO's own compute_gamma uses, just grouped by "
                        "ticker alone instead of (ticker,cls,sign). CAUTION: all of this is an IN-SAMPLE "
                        "statistic (same window it is measured on) - treat a high win_rate/pnl_lb/gamma at "
                        "low `total` as a candidate to watch, not a proven filter, until it is recomputed "
                        "walk-forward over a long enough history.",
            "lambda_basis": "per-ticker std(ddof=1) of `score` (=min(|tech_z|,|rel_z|)) across EVERY "
                        "decision-minute reading, gate-free, same population as alpha - this strategy's "
                        "own stand-in for Reversal's 'raw entry-checkpoint reading' (VWAP Bounce has no "
                        "single clean entry print to measure dispersion of).",
            "sim_params": {
                "session_open_min": SESSION_OPEN_MIN, "session_close_min": SESSION_CLOSE_MIN,
                "bucket_min": BUCKET_MIN, "min_minutes_after_open": MIN_MINUTES_AFTER_OPEN,
                "gate_z": GATE_Z, "clip_z": CLIP_Z, "target_stop_mult": TARGET_STOP_MULT,
                "add_mults": list(ADD_MULTS), "min_add_gap_min": MIN_ADD_GAP_MIN,
                "cost_bps": COST_RET * 10000, "top_fraction": TOP_FRACTION, "wscore_weights": WSCORE_W,
                "sigma_lookback_days": 20, "beta_lookback_days": BETA_LOOKBACK_DAYS,
                "beta_min_days": BETA_MIN_DAYS, "min_adv20_usd": 25_000_000,
            },
        },
        "rows": rows,
    }, f, ensure_ascii=False)

print("written ->", OUT_DIR, "(", "vwapbounce_ratings.csv", ROLLING_PATH.name, ")")
